In [1]:
# ============================================================
# D1 – Stage 4 Validation Branch C – Deterministic Normalisation
# 0. Imports and frozen validation configuration
# ============================================================

import json
import hashlib
import pandas as pd
import numpy as np

from pathlib import Path
from google.colab import files

DOCUMENT_ID = "D1"
BRANCH_ID = "C"
PARENT_BRANCH = "B"

EXPECTED_FIELDS = [
    "Geographic Area",
    "Main Occupation Group",
    "Occupation Group (2 digit)",
    "Labour Shortage Index",
    "LSI (Comp.)",
    "LSI1",
    "LSI2",
    "LSI3"
]

# Frozen from D1 Branch A validation.
MATCH_KEY_FIELDS = [
    "Geographic Area",
    "Occupation Group (2 digit)"
]

NUMERIC_FIELDS = [
    "Labour Shortage Index",
    "LSI1",
    "LSI2",
    "LSI3"
]

# LSI (Comp.) has its own conservative comparison normalisation below.
TEXT_FIELDS = [
    "Geographic Area",
    "Main Occupation Group",
    "Occupation Group (2 digit)"
]

OUTPUT_DIR = Path("outputs_D1_validation_branch_C")
OUTPUT_DIR.mkdir(exist_ok=True)

print("D1 Branch C validation configured.")

D1 Branch C validation configured.


In [2]:
# ------------------------------------------------------------
# 1. Upload validation inputs
# ------------------------------------------------------------
# Required:
#   1) D1_reference_values.csv                      [Stage 1]
#   2) D1_branch_C_parsed_extraction.json          [Branch C]
#   3) D1_branch_C_structure_check.json             [Branch C]
#   4) D1_branch_C_normalisation_check.json         [Branch C]
#
# The structure-check artefact determines output/schema validity.
# The normalisation-check artefact determines whether the Branch C
# representation preserved the Branch B parent information as intended.
#
# Representation integrity and extraction correctness remain separate outcomes.

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [f for f in uploaded_files if f.lower().endswith(".csv")]
json_files = [f for f in uploaded_files if f.lower().endswith(".json")]

if len(csv_files) != 1:
    raise ValueError("Upload exactly one Stage 1 reference-values CSV.")

if len(json_files) != 3:
    raise ValueError(
        "Upload exactly three JSON files: the Branch C parsed extraction, "
        "structure check, and normalisation check."
    )

REFERENCE_FILE = csv_files[0]

parsed_extraction_file = None
structure_check_file = None
normalisation_check_file = None

for file_name in json_files:
    with open(file_name, "r", encoding="utf-8") as f:
        obj = json.load(f)

    if isinstance(obj, dict) and isinstance(obj.get("records"), list):
        parsed_extraction_file = file_name

    if (
        isinstance(obj, dict)
        and "json_valid" in obj
        and "top_level_checks" in obj
        and "record_structure_issues" in obj
    ):
        structure_check_file = file_name

    if (
        isinstance(obj, dict)
        and "normalisation_integrity_passed" in obj
        and "parent_equivalence_passed" in obj
        and "normalised_record_count" in obj
    ):
        normalisation_check_file = file_name

if parsed_extraction_file is None:
    raise ValueError("Could not identify the D1 Branch C parsed extraction JSON.")

if structure_check_file is None:
    raise ValueError("Could not identify the D1 Branch C structure-check JSON.")

if normalisation_check_file is None:
    raise ValueError("Could not identify the D1 Branch C normalisation-check JSON.")

print("Reference values:", REFERENCE_FILE)
print("Parsed extraction:", parsed_extraction_file)
print("Structure check:", structure_check_file)
print("Normalisation check:", normalisation_check_file)

Saving D1_branch_C_parsed_extraction.json to D1_branch_C_parsed_extraction.json
Saving D1_branch_C_normalisation_check.json to D1_branch_C_normalisation_check.json
Saving D1_branch_C_structure_check.json to D1_branch_C_structure_check.json
Saving D1_reference_values.csv to D1_reference_values.csv
Reference values: D1_reference_values.csv
Parsed extraction: D1_branch_C_parsed_extraction.json
Structure check: D1_branch_C_structure_check.json
Normalisation check: D1_branch_C_normalisation_check.json


In [3]:
# ------------------------------------------------------------
# 2. Load inputs and verify document/branch identity
# ------------------------------------------------------------

with open(parsed_extraction_file, "r", encoding="utf-8") as f:
    extraction_json = json.load(f)

with open(structure_check_file, "r", encoding="utf-8") as f:
    structure_check = json.load(f)

with open(normalisation_check_file, "r", encoding="utf-8") as f:
    normalisation_check = json.load(f)

df_ref = pd.read_csv(REFERENCE_FILE)
df_ext_raw = pd.DataFrame(extraction_json["records"])

for artefact_name, artefact in {
    "parsed extraction": extraction_json,
    "structure check": structure_check,
    "normalisation check": normalisation_check
}.items():
    if artefact.get("document_id") != DOCUMENT_ID:
        raise ValueError(
            f"Unexpected {artefact_name} document_id: {artefact.get('document_id')}"
        )
    if artefact.get("branch") != BRANCH_ID:
        raise ValueError(
            f"Unexpected {artefact_name} branch: {artefact.get('branch')}"
        )

if normalisation_check.get("parent_branch") != PARENT_BRANCH:
    raise ValueError(
        f"Unexpected Branch C parent: {normalisation_check.get('parent_branch')}"
    )

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

input_provenance = {
    "reference_file": REFERENCE_FILE,
    "reference_sha256": sha256_file(REFERENCE_FILE),
    "parsed_extraction_file": parsed_extraction_file,
    "parsed_extraction_sha256": sha256_file(parsed_extraction_file),
    "structure_check_file": structure_check_file,
    "structure_check_sha256": sha256_file(structure_check_file),
    "normalisation_check_file": normalisation_check_file,
    "normalisation_check_sha256": sha256_file(normalisation_check_file)
}

print("Reference shape:", df_ref.shape)
print("Extraction shape:", df_ext_raw.shape)

Reference shape: (156, 8)
Extraction shape: (156, 8)


In [4]:
# ------------------------------------------------------------
# 3. Reconstruct schema validity and Branch C representation integrity
# ------------------------------------------------------------
# Record-count agreement is deliberately NOT part of schema validity.
# It is evaluated later as extraction completeness/performance.

top = structure_check.get("top_level_checks", {})

required_top_level_checks = [
    "output_is_json_object",
    "document_id_present",
    "document_id_correct",
    "branch_present",
    "branch_correct",
    "records_present",
    "records_is_list"
]

top_level_valid = all(bool(top.get(check, False)) for check in required_top_level_checks)

schema_validity = bool(
    structure_check.get("json_valid", False)
    and top_level_valid
    and int(structure_check.get("records_with_structure_issues", 0)) == 0
    and int(structure_check.get("numeric_field_type_issues", 0)) == 0
)

schema_diagnostics = {
    "json_valid": bool(structure_check.get("json_valid", False)),
    "top_level_valid": top_level_valid,
    "records_with_structure_issues":
        int(structure_check.get("records_with_structure_issues", 0)),
    "numeric_field_type_issues":
        int(structure_check.get("numeric_field_type_issues", 0)),
    "schema_validity": schema_validity
}

representation_integrity = {
    "parent_branch": normalisation_check.get("parent_branch"),
    "parent_equivalence_passed":
        bool(normalisation_check.get("parent_equivalence_passed", False)),
    "normalisation_integrity_passed":
        bool(normalisation_check.get("normalisation_integrity_passed", False)),
    "record_count_preserved":
        bool(normalisation_check.get("record_count_preserved", False)),
    "normalised_record_count":
        int(normalisation_check.get("normalised_record_count", 0)),
    "source_row_identity_and_order_preserved":
        bool(normalisation_check.get("source_row_identity_and_order_preserved", False)),
    "normalised_observation_key_preserved":
        bool(normalisation_check.get("normalised_observation_key_preserved", False)),
    "observation_order_and_identity_preserved":
        bool(normalisation_check.get("observation_order_and_identity_preserved", False)),
    "geographic_area_set_preserved":
        bool(normalisation_check.get("geographic_area_set_preserved", False)),
    "duplicate_observation_keys":
        int(normalisation_check.get("duplicate_observation_keys", 0)),
    "invalid_lsi_component_formats":
        int(normalisation_check.get("invalid_lsi_component_formats", 0)),
    "inconsistent_lsi_components":
        int(normalisation_check.get("inconsistent_lsi_components", 0)),
    "semantic_label_mapping_applied":
        bool(normalisation_check.get("semantic_label_mapping_applied", False)),
    "value_rounding_applied":
        bool(normalisation_check.get("value_rounding_applied", False)),
    "derived_calculation_applied":
        bool(normalisation_check.get("derived_calculation_applied", False)),
    "reference_values_used_for_transformation":
        bool(normalisation_check.get("reference_values_used_for_transformation", False))
}

print("Schema diagnostics:")
print(json.dumps(schema_diagnostics, indent=2))

print("\nBranch C representation-integrity diagnostics:")
print(json.dumps(representation_integrity, indent=2))

if not representation_integrity["normalisation_integrity_passed"]:
    print(
        "WARNING: Branch C normalisation integrity did not pass. "
        "Extraction validation will still describe the observed model output, "
        "but Stage 5 interpretation must distinguish representation loss "
        "from LLM extraction error."
    )

if representation_integrity["reference_values_used_for_transformation"]:
    raise ValueError(
        "Branch C reports use of reference values during transformation, "
        "which would violate the experimental design."
    )

Schema diagnostics:
{
  "json_valid": true,
  "top_level_valid": true,
  "records_with_structure_issues": 0,
  "numeric_field_type_issues": 0,
  "schema_validity": true
}

Branch C representation-integrity diagnostics:
{
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "record_count_preserved": true,
  "normalised_record_count": 156,
  "source_row_identity_and_order_preserved": true,
  "normalised_observation_key_preserved": true,
  "observation_order_and_identity_preserved": true,
  "geographic_area_set_preserved": true,
  "duplicate_observation_keys": 0,
  "invalid_lsi_component_formats": 0,
  "inconsistent_lsi_components": 0,
  "semantic_label_mapping_applied": false,
  "value_rounding_applied": false,
  "derived_calculation_applied": false,
  "reference_values_used_for_transformation": false
}


In [5]:
# ------------------------------------------------------------
# 4. Verify reference and extraction fields
# ------------------------------------------------------------

missing_reference_fields = [
    field for field in EXPECTED_FIELDS
    if field not in df_ref.columns
]

if missing_reference_fields:
    raise ValueError(
        f"Stage 1 reference dataset is missing fields: {missing_reference_fields}"
    )

missing_extraction_columns = [
    field for field in EXPECTED_FIELDS
    if field not in df_ext_raw.columns
]

# The preserved raw extraction is never modified.
# A validation copy is created for comparison.
df_ext = df_ext_raw.copy()

# Missing fields are represented as null only in the validation copy,
# allowing content diagnostics to continue while schema validity remains
# determined by the Branch C structure-check artefact.
for field in missing_extraction_columns:
    df_ext[field] = np.nan

df_ref = df_ref[EXPECTED_FIELDS].copy()
df_ext = df_ext[EXPECTED_FIELDS].copy()

print("Missing extraction columns:", missing_extraction_columns)

Missing extraction columns: []


In [6]:
# ------------------------------------------------------------
# 5. Deterministic comparison normalisation
# ------------------------------------------------------------
# FROZEN FROM D1 BRANCH A VALIDATION.
# These transformations are applied only to comparison copies.
# They do NOT alter the preserved Branch C model output.
#
# Note: Branch C input normalisation and Stage 4 validation normalisation
# are different operations. The rules below are the same rules used for
# Branches A and B so that branch comparison remains fair.

def normalize_text(value):
    if pd.isna(value):
        return ""

    value = str(value).strip().lower()
    value = (
        value
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )
    return " ".join(value.split())


def normalize_number(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    value = str(value).strip()
    if value == "":
        return np.nan

    # D1 contains small numerical values and no thousands separators.
    # Decimal-comma harmonisation is retained for comparison robustness.
    if "," in value and "." not in value:
        value = value.replace(",", ".")

    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def numbers_match(reference_value, extracted_value, decimals=None, tolerance=None):
    ref_num = normalize_number(reference_value)
    ext_num = normalize_number(extracted_value)

    if pd.isna(ref_num) and pd.isna(ext_num):
        return True

    if pd.isna(ref_num) or pd.isna(ext_num):
        return False

    if decimals is not None:
        return round(ref_num, decimals) == round(ext_num, decimals)

    return abs(ref_num - ext_num) <= tolerance


def normalize_lsi_comp(value):
    if pd.isna(value):
        return ""

    value = str(value).strip()
    value = (
        value
        .replace(" ", "")
        .replace("–", "-")
        .replace("—", "-")
    )
    return value


# Numerical comparison follows the precision represented in the source.
# Labour Shortage Index is compared at two decimal places.
# Integer component fields are compared exactly.
NUMERIC_RULES = {
    "Labour Shortage Index": {"decimals": 2},
    "LSI1": {"tolerance": 0.0},
    "LSI2": {"tolerance": 0.0},
    "LSI3": {"tolerance": 0.0}
}

In [7]:
# ------------------------------------------------------------
# 6. Create document-specific observation keys
# ------------------------------------------------------------
# FROZEN FROM D1 BRANCH A VALIDATION.

def create_key(dataframe):
    return (
        dataframe["Geographic Area"].apply(normalize_text)
        + " | "
        + dataframe["Occupation Group (2 digit)"].apply(normalize_text)
    )

df_ref["match_key"] = create_key(df_ref)
df_ext["match_key"] = create_key(df_ext)

reference_duplicate_count = int(df_ref["match_key"].duplicated(keep=False).sum())

if reference_duplicate_count > 0:
    raise ValueError(
        "The fixed Stage 1 reference dataset contains duplicate observation keys; "
        "the selected D1 matching key is therefore not unique."
    )

# For extracted duplicates, preserve the first occurrence for deterministic
# one-to-one alignment and classify later occurrences as unsupported duplicate outputs.
extraction_duplicate_mask = df_ext["match_key"].duplicated(keep="first")
duplicate_extraction_records = df_ext[extraction_duplicate_mask].copy()
df_ext_unique = df_ext[~extraction_duplicate_mask].copy()

print("Reference duplicate observations:", reference_duplicate_count)
print("Additional extracted duplicate observations:", len(duplicate_extraction_records))

Reference duplicate observations: 0
Additional extracted duplicate observations: 0


In [8]:
# ------------------------------------------------------------
# 7. One-to-one record alignment
# ------------------------------------------------------------

df_validation = df_ref.merge(
    df_ext_unique,
    on="match_key",
    how="outer",
    suffixes=("_ref", "_ext"),
    indicator=True,
    validate="one_to_one"
)

print(df_validation["_merge"].value_counts(dropna=False))

_merge
both          156
left_only       0
right_only      0
Name: count, dtype: int64


In [9]:
# ------------------------------------------------------------
# 8. Field-level comparison for aligned observations
# ------------------------------------------------------------

matched_mask = df_validation["_merge"] == "both"

for field in NUMERIC_FIELDS:
    col = f"{field}_match"
    rule = NUMERIC_RULES[field]

    df_validation[col] = False
    df_validation.loc[matched_mask, col] = df_validation.loc[matched_mask].apply(
        lambda row: numbers_match(
            row[f"{field}_ref"],
            row[f"{field}_ext"],
            **rule
        ),
        axis=1
    )

df_validation["LSI (Comp.)_match"] = False
df_validation.loc[matched_mask, "LSI (Comp.)_match"] = df_validation.loc[
    matched_mask
].apply(
    lambda row:
        normalize_lsi_comp(row["LSI (Comp.)_ref"])
        == normalize_lsi_comp(row["LSI (Comp.)_ext"]),
    axis=1
)

for field in TEXT_FIELDS:
    col = f"{field}_match"
    df_validation[col] = False
    df_validation.loc[matched_mask, col] = df_validation.loc[matched_mask].apply(
        lambda row:
            normalize_text(row[f"{field}_ref"])
            == normalize_text(row[f"{field}_ext"]),
        axis=1
    )

MATCH_COLUMNS = [f"{field}_match" for field in EXPECTED_FIELDS]

df_validation["all_fields_match"] = (
    matched_mask
    & df_validation[MATCH_COLUMNS].all(axis=1)
)

In [10]:
# ------------------------------------------------------------
# 9. Classify record outcomes
# ------------------------------------------------------------

def classify_record(row):
    if row["_merge"] == "left_only":
        return "missing"
    if row["_merge"] == "right_only":
        return "hallucinated_unsupported"
    if bool(row["all_fields_match"]):
        return "fully_correct"
    return "discrepant"

df_validation["record_status"] = df_validation.apply(classify_record, axis=1)

missing_records = df_validation[
    df_validation["record_status"] == "missing"
].copy()

unsupported_records = df_validation[
    df_validation["record_status"] == "hallucinated_unsupported"
].copy()

discrepant_records = df_validation[
    df_validation["record_status"] == "discrepant"
].copy()

fully_correct_records_df = df_validation[
    df_validation["record_status"] == "fully_correct"
].copy()

# Duplicate extra outputs are unsupported extra records.
duplicate_extraction_records["record_status"] = "hallucinated_duplicate"

print("Missing:", len(missing_records))
print("Unsupported unmatched:", len(unsupported_records))
print("Unsupported duplicate extras:", len(duplicate_extraction_records))
print("Discrepant matched:", len(discrepant_records))
print("Fully correct:", len(fully_correct_records_df))

Missing: 0
Unsupported unmatched: 0
Unsupported duplicate extras: 0
Discrepant matched: 0
Fully correct: 156


In [11]:
# ------------------------------------------------------------
# 10. Calculate record-level and field-level metrics
# ------------------------------------------------------------

N_REF = int(len(df_ref))
N_EXT = int(len(df_ext))
N_ALIGNED = int((df_validation["_merge"] == "both").sum())
N_MISSING = int(len(missing_records))
N_UNSUPPORTED_UNMATCHED = int(len(unsupported_records))
N_DUPLICATE_EXTRAS = int(len(duplicate_extraction_records))
N_HALLUCINATED = N_UNSUPPORTED_UNMATCHED + N_DUPLICATE_EXTRAS
N_DISCREPANT = int(len(discrepant_records))
N_CORRECT = int(len(fully_correct_records_df))

# Completeness measures whether expected observations were returned,
# independently of whether every extracted value is correct.
completeness = N_ALIGNED / N_REF if N_REF else 0.0

# Exact-record metrics:
# a record is correct only when ALL expected fields agree.
record_precision = N_CORRECT / N_EXT if N_EXT else 0.0
record_recall = N_CORRECT / N_REF if N_REF else 0.0
record_f1 = (
    2 * record_precision * record_recall / (record_precision + record_recall)
    if (record_precision + record_recall) > 0 else 0.0
)

hallucination_rate = N_HALLUCINATED / N_EXT if N_EXT else 0.0
discrepancy_rate = N_DISCREPANT / N_ALIGNED if N_ALIGNED else 0.0

matched_validation = df_validation[df_validation["_merge"] == "both"].copy()

field_accuracy_matched = {}
for field in EXPECTED_FIELDS:
    col = f"{field}_match"
    field_accuracy_matched[field] = (
        float(matched_validation[col].mean())
        if len(matched_validation) else 0.0
    )

# Overall field accuracy counts missing expected records as incorrect field instances.
correct_field_instances = int(
    matched_validation[MATCH_COLUMNS].sum().sum()
)
expected_field_instances = int(N_REF * len(EXPECTED_FIELDS))
overall_field_accuracy = (
    correct_field_instances / expected_field_instances
    if expected_field_instances else 0.0
)

summary = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "parent_branch": PARENT_BRANCH,
    "reference_records": N_REF,
    "extracted_records": N_EXT,
    "aligned_records": N_ALIGNED,
    "fully_correct_records": N_CORRECT,
    "discrepant_records": N_DISCREPANT,
    "missing_records": N_MISSING,

    # Internal naming is retained for comparability with Branch A/B notebooks.
    # These correspond to unsupported extra records in the dissertation tables.
    "hallucinated_records": N_HALLUCINATED,
    "hallucinated_unmatched_records": N_UNSUPPORTED_UNMATCHED,
    "hallucinated_duplicate_records": N_DUPLICATE_EXTRAS,
    "unsupported_records": N_HALLUCINATED,

    "completeness": round(completeness, 4),
    "record_precision_exact": round(record_precision, 4),
    "record_recall_exact": round(record_recall, 4),
    "record_f1_exact": round(record_f1, 4),
    "hallucination_rate": round(hallucination_rate, 4),
    "discrepancy_rate_among_aligned": round(discrepancy_rate, 4),
    "overall_field_accuracy": round(overall_field_accuracy, 4),
    "field_accuracy_among_aligned": {
        k: round(v, 4) for k, v in field_accuracy_matched.items()
    },
    "schema_validity": schema_validity,
    "schema_diagnostics": schema_diagnostics,
    "branch_C_representation_integrity": representation_integrity,
    "matching_key_fields": MATCH_KEY_FIELDS,
    "comparison_rules_frozen_from_branch_A": True,
    "reference_dataset_branch_independent": True,
    "normalisation_note":
        "Validation normalisation was applied only to comparison copies using "
        "the rules frozen in D1 Branch A validation; the preserved Branch C "
        "extraction was not modified.",
    "input_provenance": input_provenance
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "document_id": "D1",
  "branch": "C",
  "parent_branch": "B",
  "reference_records": 156,
  "extracted_records": 156,
  "aligned_records": 156,
  "fully_correct_records": 156,
  "discrepant_records": 0,
  "missing_records": 0,
  "hallucinated_records": 0,
  "hallucinated_unmatched_records": 0,
  "hallucinated_duplicate_records": 0,
  "unsupported_records": 0,
  "completeness": 1.0,
  "record_precision_exact": 1.0,
  "record_recall_exact": 1.0,
  "record_f1_exact": 1.0,
  "hallucination_rate": 0.0,
  "discrepancy_rate_among_aligned": 0.0,
  "overall_field_accuracy": 1.0,
  "field_accuracy_among_aligned": {
    "Geographic Area": 1.0,
    "Main Occupation Group": 1.0,
    "Occupation Group (2 digit)": 1.0,
    "Labour Shortage Index": 1.0,
    "LSI (Comp.)": 1.0,
    "LSI1": 1.0,
    "LSI2": 1.0,
    "LSI3": 1.0
  },
  "schema_validity": true,
  "schema_diagnostics": {
    "json_valid": true,
    "top_level_valid": true,
    "records_with_structure_issues": 0,
    "numeric_field_type

In [12]:
# ------------------------------------------------------------
# 11. Field-level error summary
# ------------------------------------------------------------

field_error_summary = []

for field in EXPECTED_FIELDS:
    col = f"{field}_match"

    correct_aligned = int(matched_validation[col].sum())
    incorrect_aligned = int(len(matched_validation) - correct_aligned)

    field_error_summary.append({
        "field": field,
        "used_in_matching_key": field in MATCH_KEY_FIELDS,
        "aligned_records_evaluated": int(len(matched_validation)),
        "correct_values_among_aligned": correct_aligned,
        "incorrect_values_among_aligned": incorrect_aligned,
        "accuracy_among_aligned":
            round(correct_aligned / len(matched_validation), 4)
            if len(matched_validation) else 0.0,
        "missing_expected_instances": N_MISSING,
        "overall_correct_instances": correct_aligned,
        "overall_expected_instances": N_REF,
        "overall_field_accuracy":
            round(correct_aligned / N_REF, 4)
            if N_REF else 0.0
    })

field_error_summary_df = pd.DataFrame(field_error_summary)
display(field_error_summary_df)

,field,used_in_matching_key,aligned_records_evaluated,correct_values_among_aligned,incorrect_values_among_aligned,accuracy_among_aligned,missing_expected_instances,overall_correct_instances,overall_expected_instances,overall_field_accuracy
0,Geographic Area,True,156,156,0,1.0,0,156,156,1.0
1,Main Occupation Group,False,156,156,0,1.0,0,156,156,1.0
2,Occupation Group (2 digit),True,156,156,0,1.0,0,156,156,1.0
3,Labour Shortage Index,False,156,156,0,1.0,0,156,156,1.0
4,LSI (Comp.),False,156,156,0,1.0,0,156,156,1.0
5,LSI1,False,156,156,0,1.0,0,156,156,1.0
6,LSI2,False,156,156,0,1.0,0,156,156,1.0
7,LSI3,False,156,156,0,1.0,0,156,156,1.0


In [13]:
# ------------------------------------------------------------
# 12. Compact validation-results table
# ------------------------------------------------------------
# This is a concise notebook-level results view for checking before export.

overall_metrics_df = pd.DataFrame([
    {"metric": "Reference records", "value": N_REF},
    {"metric": "Extracted records", "value": N_EXT},
    {"metric": "Aligned records", "value": N_ALIGNED},
    {"metric": "Fully correct records", "value": N_CORRECT},
    {"metric": "Discrepant records", "value": N_DISCREPANT},
    {"metric": "Missing records", "value": N_MISSING},
    {"metric": "Unsupported records", "value": N_HALLUCINATED},
    {"metric": "Completeness", "value": round(completeness, 4)},
    {"metric": "Exact precision", "value": round(record_precision, 4)},
    {"metric": "Exact recall", "value": round(record_recall, 4)},
    {"metric": "Exact F1", "value": round(record_f1, 4)},
    {"metric": "Overall field accuracy", "value": round(overall_field_accuracy, 4)},
    {"metric": "Hallucination/unsupported rate", "value": round(hallucination_rate, 4)},
    {"metric": "Schema validity", "value": schema_validity},
    {
        "metric": "Branch C normalisation integrity",
        "value": representation_integrity["normalisation_integrity_passed"]
    }
])

display(overall_metrics_df)

,metric,value
0,Reference records,156
1,Extracted records,156
2,Aligned records,156
3,Fully correct records,156
4,Discrepant records,0
5,Missing records,0
6,Unsupported records,0
7,Completeness,1.0
8,Exact precision,1.0
9,Exact recall,1.0


In [14]:
# ------------------------------------------------------------
# 13. Validation integrity checks
# ------------------------------------------------------------

# Every reference record must be either aligned or missing.
assert N_ALIGNED + N_MISSING == N_REF

# Every extracted record must be either a unique aligned/unmatched record
# or an additional duplicate output.
assert (
    N_ALIGNED
    + N_UNSUPPORTED_UNMATCHED
    + N_DUPLICATE_EXTRAS
    == N_EXT
)

# Every aligned record must be either fully correct or discrepant.
assert N_CORRECT + N_DISCREPANT == N_ALIGNED

# The fixed matching key must remain unique in the reference dataset.
assert reference_duplicate_count == 0

# Rates must remain in valid bounds.
for metric_name, metric_value in {
    "completeness": completeness,
    "record_precision": record_precision,
    "record_recall": record_recall,
    "record_f1": record_f1,
    "hallucination_rate": hallucination_rate,
    "discrepancy_rate": discrepancy_rate,
    "overall_field_accuracy": overall_field_accuracy
}.items():
    assert 0.0 <= metric_value <= 1.0, f"Invalid {metric_name}: {metric_value}"

print("Validation integrity checks passed.")

Validation integrity checks passed.


In [15]:
# ------------------------------------------------------------
# 14. Export validation artefacts
# ------------------------------------------------------------

df_validation.to_csv(
    OUTPUT_DIR / "D1_branch_C_validation_detailed.csv",
    index=False
)

missing_records.to_csv(
    OUTPUT_DIR / "D1_branch_C_missing_records.csv",
    index=False
)

unsupported_records.to_csv(
    OUTPUT_DIR / "D1_branch_C_hallucinated_unmatched_records.csv",
    index=False
)

duplicate_extraction_records.to_csv(
    OUTPUT_DIR / "D1_branch_C_hallucinated_duplicate_records.csv",
    index=False
)

discrepant_records.to_csv(
    OUTPUT_DIR / "D1_branch_C_discrepant_records.csv",
    index=False
)

fully_correct_records_df.to_csv(
    OUTPUT_DIR / "D1_branch_C_fully_correct_records.csv",
    index=False
)

field_error_summary_df.to_csv(
    OUTPUT_DIR / "D1_branch_C_field_error_summary.csv",
    index=False
)

overall_metrics_df.to_csv(
    OUTPUT_DIR / "D1_branch_C_overall_metrics.csv",
    index=False
)

with open(
    OUTPUT_DIR / "D1_branch_C_validation_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Validation artefacts saved.")

Validation artefacts saved.


In [16]:
# ------------------------------------------------------------
# 15. Final validation report
# ------------------------------------------------------------

final_report = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "reference_records": N_REF,
    "extracted_records": N_EXT,
    "aligned_records": N_ALIGNED,
    "fully_correct_records": N_CORRECT,
    "discrepant_records": N_DISCREPANT,
    "missing_records": N_MISSING,
    "unsupported_records": N_HALLUCINATED,
    "completeness": round(completeness, 4),
    "record_precision_exact": round(record_precision, 4),
    "record_recall_exact": round(record_recall, 4),
    "record_f1_exact": round(record_f1, 4),
    "overall_field_accuracy": round(overall_field_accuracy, 4),
    "schema_validity": schema_validity,
    "normalisation_integrity_passed":
        representation_integrity["normalisation_integrity_passed"],
    "comparison_rules_frozen_from_branch_A": True
}

print(json.dumps(final_report, indent=2, ensure_ascii=False))

{
  "document_id": "D1",
  "branch": "C",
  "reference_records": 156,
  "extracted_records": 156,
  "aligned_records": 156,
  "fully_correct_records": 156,
  "discrepant_records": 0,
  "missing_records": 0,
  "unsupported_records": 0,
  "completeness": 1.0,
  "record_precision_exact": 1.0,
  "record_recall_exact": 1.0,
  "record_f1_exact": 1.0,
  "overall_field_accuracy": 1.0,
  "schema_validity": true,
  "normalisation_integrity_passed": true,
  "comparison_rules_frozen_from_branch_A": true
}


In [17]:
# ------------------------------------------------------------
# 16. Download validation artefacts
# ------------------------------------------------------------

for output_file in sorted(OUTPUT_DIR.iterdir()):
    if output_file.is_file():
        print("Downloading:", output_file.name)
        files.download(output_file)

Downloading: D1_branch_C_discrepant_records.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: D1_branch_C_field_error_summary.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: D1_branch_C_validation_summary.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>